<a href="https://colab.research.google.com/github/kmillaevelyn/data-science-portfolio/blob/main/08-academico/bd-mysql-campeonato/bd-mysql-campeonato.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 39.2 MB/s eta 0:00:00


In [ ]:
import mysql.connector

from mysql.connector import errorcode
import random

try:
    # Conexão ao banco de dados
    conexao = mysql.connector.connect(
        host='test_db.mysql.dbaas.com.br',
        user='test_db',
        password='BD@2024bctec',
        database='test_db'
    )
    cursor = conexao.cursor()

    # Criação da tabela "partida_kejp"
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS partida_kejp (
            id INT AUTO_INCREMENT PRIMARY KEY,
            time_casa VARCHAR(50),
            gols_casa INT,
            time_visitante VARCHAR(50),
            gols_visitante INT
        );
    """)

    # Inserção de partidas
    times_rodada = [
        ("Botafogo", "Palmeiras"),
        ("Fortaleza", "Flamengo"),
        ("Vitória", "Internacional"),
        ("São Paulo", "Cruzeiro"),
        ("Bahia", "Corinthians"),
        ("Vasco da Gama", "Atlético-MG"),
        ("Grêmio", "Fluminense"),
        ("Athletico-PR", "Juventude")
    ]

    for time_casa, time_visitante in times_rodada:
        cursor.execute("""
            INSERT INTO partida_kejp (time_casa, gols_casa, time_visitante, gols_visitante)
            VALUES (%s, %s, %s, %s)
        """, (time_casa, 0, time_visitante, 0))

    # Atualização dos gols
    cursor.execute("SELECT id FROM partida_kejp")
    ids_partidas = [row[0] for row in cursor.fetchall()]

    for id_partida in ids_partidas:
        gols_casa = random.randint(0, 7)
        gols_visitante = random.randint(0, 7)
        cursor.execute("""
            UPDATE partida_kejp
            SET gols_casa = %s, gols_visitante = %s
            WHERE id = %s
        """, (gols_casa, gols_visitante, id_partida))

    # Criação da tabela "estatisticas_kejp"
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS estatisticas_kejp (
            id INT AUTO_INCREMENT PRIMARY KEY,
            id_partida INT,
            intervalo_tempo VARCHAR(20),
            amarelos_casa INT,
            amarelos_visitante INT,
            vermelhos_casa INT,
            vermelhos_visitante INT,
            faltas_casa INT,
            faltas_visitante INT,
            escanteios_casa INT,
            escanteios_visitante INT,
            impedimentos_casa INT,
            impedimentos_visitante INT,
            FOREIGN KEY (id_partida) REFERENCES partida_kejp(id)
        );
    """)

    # Inserção de estatísticas
    intervalos_tempo = [
        "0 a 10m", "11 a 20m", "21 a 30m", "31 a 40m",
        "41 a 50m", "51 a 60m", "61 a 70m", "71 a 80m", "81 a 90m"
    ]

    for id_partida in ids_partidas:
        for intervalo in intervalos_tempo:
            estatisticas = (
                id_partida,
                intervalo,
                random.randint(0, 3),  # Amarelos casa
                random.randint(0, 3),  # Amarelos visitante
                random.randint(0, 1),  # Vermelhos casa
                random.randint(0, 1),  # Vermelhos visitante
                random.randint(0, 15), # Faltas casa
                random.randint(0, 15), # Faltas visitante
                random.randint(0, 5),  # Escanteios casa
                random.randint(0, 5),  # Escanteios visitante
                random.randint(0, 3),  # Impedimentos casa
                random.randint(0, 3)   # Impedimentos visitante
            )
            cursor.execute("""
                INSERT INTO estatisticas_kejp (
                    id_partida, intervalo_tempo, amarelos_casa, amarelos_visitante,
                    vermelhos_casa, vermelhos_visitante, faltas_casa, faltas_visitante,
                    escanteios_casa, escanteios_visitante, impedimentos_casa, impedimentos_visitante
                ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, estatisticas)

    conexao.commit()

    # Consulta das estatísticas
    cursor.execute("""
        SELECT id_partida, intervalo_tempo, amarelos_casa, amarelos_visitante,
               vermelhos_casa, vermelhos_visitante, faltas_casa, faltas_visitante,
               escanteios_casa, escanteios_visitante, impedimentos_casa, impedimentos_visitante
        FROM estatisticas_kejp
        ORDER BY id_partida, intervalo_tempo
    """)
    todas_estatisticas = cursor.fetchall()

    print("\nEstatísticas das Partidas:")
    partida_atual = None
    for estatistica in todas_estatisticas:
        (id_partida, intervalo_tempo, amarelos_casa, amarelos_visitante,
         vermelhos_casa, vermelhos_visitante, faltas_casa, faltas_visitante,
         escanteios_casa, escanteios_visitante, impedimentos_casa, impedimentos_visitante) = estatistica

        if id_partida != partida_atual:
            print(f"\nPartida {id_partida}:")
            partida_atual = id_partida

        print(f" Intervalo {intervalo_tempo}:")
        print(f"  - Cartões amarelos: {amarelos_casa} (Casa) x {amarelos_visitante} (Visitante)")
        print(f"  - Cartões vermelhos: {vermelhos_casa} (Casa) x {vermelhos_visitante} (Visitante)")
        print(f"  - Faltas: {faltas_casa} (Casa) x {faltas_visitante} (Visitante)")
        print(f"  - Escanteios: {escanteios_casa} (Casa) x {escanteios_visitante} (Visitante)")
        print(f"  - Impedimentos: {impedimentos_casa} (Casa) x {impedimentos_visitante} (Visitante)")

except mysql.connector.Error as err:
    if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
        print("Erro: Usuário ou senha inválidos")
    elif err.errno == errorcode.ER_BAD_DB_ERROR:
        print("Erro: Banco de dados não encontrado")
    else:
        print(f"Erro: {err}")
finally:
    if conexao.is_connected():
        cursor.close()
        conexao.close()

A saída de streaming foi truncada nas últimas 5000 linhas.
 Intervalo 61 a 70m:
  - Cartões amarelos: 1 (Casa) x 1 (Visitante)
  - Cartões vermelhos: 1 (Casa) x 1 (Visitante)
  - Faltas: 13 (Casa) x 3 (Visitante)
  - Escanteios: 3 (Casa) x 0 (Visitante)
  - Impedimentos: 0 (Casa) x 1 (Visitante)
 Intervalo 61 a 70m:
  - Cartões amarelos: 0 (Casa) x 0 (Visitante)
  - Cartões vermelhos: 1 (Casa) x 1 (Visitante)
  - Faltas: 14 (Casa) x 6 (Visitante)
  - Escanteios: 3 (Casa) x 1 (Visitante)
  - Impedimentos: 2 (Casa) x 2 (Visitante)
 Intervalo 71 a 80m:
  - Cartões amarelos: 4 (Casa) x 1 (Visitante)
  - Cartões vermelhos: 2 (Casa) x 1 (Visitante)
  - Faltas: 23 (Casa) x 52 (Visitante)
  - Escanteios: 7 (Casa) x 10 (Visitante)
  - Impedimentos: 17 (Casa) x 27 (Visitante)
 Intervalo 71 a 80m:
  - Cartões amarelos: 2 (Casa) x 3 (Visitante)
  - Cartões vermelhos: 2 (Casa) x 3 (Visitante)
  - Faltas: 56 (Casa) x 24 (Visitante)
  - Escanteios: 8 (Casa) x 7 (Visitante)
  - Impedimentos: 15 (Casa)